# 完整可用训练历史的固定恢复检验
纳入冻结来源最早完整53日，不挑日期凑样本。最多3配置+3恢复拟合，1训练参考+6配对原生账户；参数、门槛和2020–2022原始配置预测不变，双方同预热/同现金起点。

In [1]:
from pathlib import Path
from dataclasses import asdict
import os,json,hashlib,warnings
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.exceptions import ConvergenceWarning
from src.workflow.minute_action_value import _source
from src.workflow.minute_pit_allocator import _primary_dataset,_auxiliary_dataset,_prediction_dates,_ensemble,_dataset
from src.workflow.pit_constituent_auxiliary_training_allocation import _smooth_weights
from src.workflow.stage1_open_action_value import Stage1OpenActionValueSpec
from src.workflow.intraday_recovery_direct_allocation import IntradayRecoveryDirectAllocationSpec,FEATURE_COLUMNS
from src.model.tangent_allocator import TangentHeadAllocator
from src.model.annual_recovery import AnnualRecoveryControllers
from src.data.recovery_first_hour import FIELDS
from src.analysis.recovery_training_labels import recovery_training_labels
from src.backtest.full_path_tiered import replay_full_path
from src.backtest.stage1_ibkr import native_tiered_stage1
from src.backtest.qlib_continuous_position import QlibContinuousPosition
from src.analysis.integrated_allocation import paired_metrics,stationary_bootstrap_excess
from src.data.nas_market_data import archive_nas_recorder
from src.data.minute_intake import atomic_json
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/full_history_recovery_20261015';p=json.loads((card/'plan.json').read_text());provider=root/'data/qlib/full_path_tiered_allocation_20261013';os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(provider),region='us')
training,ta=_source(root,p['training_source'],['inputs/daily_frame','inputs/features','inputs/common','inputs/timing','inputs/native_prices','inputs/auxiliary','inputs/plan',*[f'models/base_2020_{seed}' for seed in p['base_seeds']]])
market,ma=_source(root,p['price_source'],['inputs/frame','inputs/plan']);early,ea=_source(root,p['early_source'],['analysis/oof_predictions','models/2019_71']);later,la=_source(root,p['later_source'],['analysis/oof_predictions']);hours,ha=_source(root,p['hour_source'],['inputs/hour_features','inputs/hour_manifest','inputs/selection']);prior,pa=_source(root,p['prior_account_source'],['inputs/frame',*[f'analysis/native/{family}_{cost}' for family in ['control','candidate'] for cost in p['slippage_bps']]]);bench,ba=_source(root,p['benchmark_source'],['inputs/frame',*[f'analysis/native/soxx_{cost}' for cost in p['slippage_bps']]]);action,aa=_source(root,p['action_source'],['inputs/plan'])
assert market['inputs/plan']['tariff']==p['tariff'];assert hours['inputs/hour_manifest']['source_manifest_sha256']==market['inputs/plan']['source_id']
base=training['inputs/daily_frame'];calendar=pd.DatetimeIndex(training['inputs/native_prices'].xs('SOXX',level='instrument').index);purge=training['inputs/plan']['purge_rows'];assert calendar[calendar.searchsorted(pd.Timestamp('2019-01-01'))-purge-1]==early['models/2019_71'].train_end
train_end=calendar[calendar.searchsorted(pd.Timestamp('2018-01-01'))-purge-1];native_early=market['inputs/frame'][market['inputs/frame'].trade_date.dt.year.eq(2018)];dates=pd.DatetimeIndex(native_early.signal_date);assert len(dates)==53
original=training['models/base_2020_71'];inputs={'frame':base,'features':training['inputs/features'],'common':training['inputs/common'],'timing':training['inputs/timing'],'spec':original.spec};dataset=_primary_dataset(inputs,train_end=str(train_end.date()),predict_bounds=(str(dates.min().date()),str(dates.max().date())));auxiliary=_auxiliary_dataset(training['inputs/auxiliary']['2020'],str(train_end.date()));assert len(auxiliary.prepare('train'))==0
state_path=card/'run_state.json';digest=hashlib.sha256((card/'plan.json').read_bytes()).hexdigest();state=json.loads(state_path.read_text()) if state_path.exists() else {'plan_sha256':digest,'fits':[],'native':[]};assert state['plan_sha256']==digest
kwargs={'experiment_name':card.name,'uri':str(card/'qlib_recorder')}
if state.get('recorder_id'):kwargs.update(recorder_id=state['recorder_id'],resume=True)
raw={};fit_audit=[]
with R.start(**kwargs):
 rec=R.get_recorder();state.update(recorder_id=rec.id,experiment_id=str(rec.experiment_id));atomic_json(state,state_path);rec.save_objects(artifact_path='inputs',plan=p,source_audits={'training':ta,'market':ma,'early':ea,'later':la,'hours':ha,'prior':pa,'benchmark':ba,'action':aa});rec.log_params(maximum_model_fits=6,maximum_new_native_cases=7)
 for seed in p['base_seeds']:
  name=f'allocation2018_{seed}';template=training[f'models/base_2020_{seed}']
  if name in state['fits']:model=rec.load_object('models/'+name)
  else:
   assert len(state['fits'])<p['maximum_model_fits'];model=TangentHeadAllocator(base_features=template.base_features,extra_features=[],feature_masks=template.feature_masks,spec={**asdict(template.spec),'root':str(root)},seed=seed+2018,cutoff='2018-01-01 00:00:00-05:00',train_end=str(train_end.date()),minimum_observations=template.minimum_observations);model.fit(dataset,auxiliary_dataset=auxiliary);rec.save_objects(artifact_path='models',**{name:model});state['fits'].append(name);atomic_json(state,state_path)
  assert pd.Timestamp(model.audit['last_mature_label_at'])<pd.Timestamp('2018-01-01',tz='America/New_York');pred=_prediction_dates(model.predict(dataset));assert pred.index.equals(dates);raw[('2018',seed)]=pred;fit_audit.append({'family':'allocation','year':2018,'base_seed':seed,**model.audit})
 initial=_ensemble(raw,'2018',p['base_seeds']);initial['trade_date']=base.loc[initial.index,'trade_date'];initial['model_year']=2018;rec.save_objects(artifact_path='analysis',early_predictions=initial,early_training=fit_audit)
pred=pd.concat([initial,early['analysis/oof_predictions'],later['analysis/oof_predictions']]).sort_index();assert len(pred)==1061 and not pred.index.duplicated().any();returns=base.loc[pred.index,['soxl_forward_open_return','soxx_forward_open_return']];returns.columns=['soxl_return','soxx_return'];targets=_smooth_weights(pred.drop(columns=['trade_date','model_year']),spec=original.spec,candidate=original.candidate,asset_returns=returns)
frame=market['inputs/frame'].set_index('signal_date').reindex(pred.index).rename_axis('signal_date').reset_index();assert pd.DatetimeIndex(frame.trade_date).equals(pd.DatetimeIndex(pred.trade_date));hour=hours['inputs/hour_features'].reindex(pd.DatetimeIndex(frame.trade_date));assert hour.first_hour_ready.notna().all()
for field in FIELDS:frame[field]=hour[field].to_numpy()
frame['first_hour_ready']=hour.first_hour_ready.to_numpy();frame['recovery_feature_available_at']=hour.recovery_feature_available_at.to_numpy()
for field in targets:frame[field]=targets[field].to_numpy()
selection=hours['inputs/selection'];recovery=IntradayRecoveryDirectAllocationSpec(root=root,**action['inputs/plan']['recovery_spec']);spec=Stage1OpenActionValueSpec(root=root,**action['inputs/plan']['action_spec']);no_recovery={'frozen_open_action_value_model':None,'frozen_recovery_model':None}
def native_case(key,local,cost,**controller):
 if key in state['native']:return rec.load_object('analysis/native/'+key)
 assert len(state['native'])<p['maximum_new_native_cases'];replay=replay_full_path(local,models=None,plan=p,slippage_bps=cost,recovery_spec=recovery,action_spec=spec,selection=selection,**controller);book=native_tiered_stage1(local,replay,provider_uri=provider,tariff=p['tariff'],slippage_bps=cost,initial_cash=p['initial_cash'],freq='1min',opening_offset=pd.Timedelta(hours=9,minutes=47),position_type=QlibContinuousPosition);book['targets']=replay['targets'];rec.save_objects(artifact_path='analysis/native',**{key:book});state['native'].append(key);atomic_json(state,state_path);rec.log_metrics(completed_native_cases=len(state['native']));return book
reference=native_case('training_reference',frame,2,controller=no_recovery);labels=recovery_training_labels(frame,reference,plan=p,recovery_spec=recovery,action_spec=spec,selection=selection,slippage_bps=2);x=labels.set_index('trade_date');ds=_dataset(x[list(FEATURE_COLUMNS)],x[['net_advantage']],pd.DataFrame(index=x.index),x[['eligible','label_available_at']],{str(y):(str(frame.trade_date.min().date()),f'{y-1}-12-31') for y in [2020,2021,2022]});audits=[];models={}
with R.start(experiment_name=card.name,uri=str(card/'qlib_recorder'),recorder_id=rec.id,resume=True):
 for year in [2020,2021,2022]:
  a=ds.prepare(str(year),col_set=['feature','label','meta']);cutoff=pd.Timestamp(f'{year}-01-01',tz='America/New_York');valid=a[('meta','eligible')].astype(bool)&pd.to_datetime(a[('meta','label_available_at')]).lt(cutoff)&np.isfinite(a[('label','net_advantage')])&np.isfinite(a['feature']).all(axis=1);train=a.loc[valid];y=train[('label','net_advantage')].gt(0);audit={'year':year,'rows':len(train),'positive_rows':int(y.sum()),'minimum_rows':20,'cutoff':str(cutoff),'last_label_available_at':str(train[('meta','label_available_at')].max()),'status':'needs_data'}
  if len(train)>=20 and y.nunique()==2:
   name=f'recovery{year}'
   if name in state['fits']:model=rec.load_object('models/'+name)
   else:
    assert len(state['fits'])<p['maximum_model_fits'];model=make_pipeline(SimpleImputer(),StandardScaler(),LogisticRegression(C=selection['logistic_c'],class_weight='balanced',max_iter=2000,random_state=42))
    with warnings.catch_warnings(record=True) as caught:
     warnings.simplefilter('always',ConvergenceWarning);model.fit(train['feature'],y)
     if any(issubclass(w.category,ConvergenceWarning) for w in caught):raise ValueError('fixed model failed convergence')
    rec.save_objects(artifact_path='models',**{name:model});state['fits'].append(name);atomic_json(state,state_path)
   models[year]=model;audit.update(status='fitted',iterations=model[-1].n_iter_.tolist())
  audits.append(audit);rec.log_metrics(completed_fits=len(state['fits']))
 rec.save_objects(artifact_path='analysis',labels=labels,training_audit=audits);rec.save_objects(artifact_path='inputs',full_reference_frame=frame,selection=selection)
policy=AnnualRecoveryControllers(models,pd.DataFrame(audits));evaluation=frame[frame.trade_date.between(p['evaluation_first_trade'],p['evaluation_last_trade'])].reset_index(drop=True);assert len(evaluation)==756;assert evaluation.trade_date.equals(prior['inputs/frame'].trade_date.reset_index(drop=True))
for col in ['soxl_open','soxx_open','raw_soxl_open','raw_soxx_open','soxl_exit_open','soxx_exit_open']:np.testing.assert_allclose(evaluation[col],bench['inputs/frame'][col].to_numpy(),rtol=0,atol=0)
w=['soxl_weight','soxx_weight','cash_weight'];warm_difference=float(np.max(np.abs(evaluation[w].to_numpy()-prior['inputs/frame'][w].to_numpy())))
books={}
for cost in p['slippage_bps']:
 books[('control',cost)]=native_case(f'control_{cost}',evaluation,cost,controller=no_recovery);books[('candidate',cost)]=native_case(f'candidate_{cost}',evaluation,cost,controller_by_date=policy);books[('prior_candidate',cost)]=prior[f'analysis/native/candidate_{cost}'];books[('soxx',cost)]=bench[f'analysis/native/soxx_{cost}']
rows=[];annual=[];uncertainty=[]
for (family,cost),book in books.items():
 path=book['authority'];b=books[('soxx',cost)]['authority'];relative=np.log1p(path.strategy_return)-np.log1p(b.strategy_return);rows.append({'family':family,'slippage_bps':cost,**paired_metrics(path,b),'monitored_drawdown_pct':-100*path.monitored_drawdown.max(),'recovery_actions':int(path.recovery_action.sum()),'positive_126d_fraction':relative.rolling(126).sum().dropna().gt(0).mean()})
 for year in [2020,2021,2022]:
  mask=path.trade_date.dt.year.eq(year);annual.append({'family':family,'slippage_bps':cost,'year':year,**paired_metrics(path.loc[mask],b.loc[mask]),'recovery_actions':int(path.loc[mask,'recovery_action'].sum())})
 if family=='candidate':
  for comparator in ['control','soxx']:uncertainty.append({'slippage_bps':cost,'comparator':comparator,**stationary_bootstrap_excess(path,books[(comparator,cost)]['authority'],mean_block=63,resamples=2000,alpha=.025,seed=20261015)})
table=pd.DataFrame(rows);annual=pd.DataFrame(annual);uncertainty=pd.DataFrame(uncertainty);cnd=table[table.family.eq('candidate')].set_index('slippage_bps');ctl=table[table.family.eq('control')].set_index('slippage_bps');gates={'beat_control_all_costs':bool(cnd.terminal_wealth.gt(ctl.terminal_wealth+1e-8).all()),'net_soxx_excess_all_costs':bool(cnd.relative_wealth.gt(1).all()),'drawdown_30pct':bool(cnd.max_drawdown_pct.ge(-30).all() and cnd.monitored_drawdown_pct.ge(-30).all()),'cvar_no_worse':bool(cnd.cvar5_pct.ge(cnd.benchmark_cvar5_pct).all()),'annual_70pct':bool(annual[annual.family.eq('candidate')].groupby('slippage_bps').relative_wealth.apply(lambda v:v.gt(1).mean()).ge(.7).all()),'rolling_70pct':bool(cnd.positive_126d_fraction.ge(.7).all()),'bootstrap_positive':bool(uncertainty.lower_bound_positive.all()),'independent_validation':False};policy_meta={'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','models/*','analysis/native/*','analysis/labels','analysis/early_predictions']};summary={'experiment':card.name,'decision':'not_promoted_full_history_recovery','gates':gates,'new_fits':len(state['fits']),'new_native_cases':len(state['native']),'earliest_training_trade':str(frame.trade_date.min().date()),'early_allocation_train_end':str(train_end.date()),'early_prediction_rows':len(initial),'all_recovery_years_available':len(models)==3,'evaluation_target_max_change_from_prior':warm_difference,'provider_uri':str(provider),'region':'us','as_of':p['as_of'],'recorder_id':rec.id,'experiment_id':str(rec.experiment_id),'recorder_uri':str(card/'qlib_recorder'),'recorder_local_dir':str(card/'qlib_recorder'/str(rec.experiment_id)/rec.id),'artifact_policy':policy_meta,'scope':p['scope'],'objective_achieved':False};summary.update(evaluation_window={'first_signal':str(books[('candidate',2)]['authority'].signal_date.min().date()),'first_trade':str(books[('candidate',2)]['authority'].trade_date.min().date()),'last_trade':str(books[('candidate',2)]['authority'].trade_date.max().date()),'last_exit':str(books[('candidate',2)]['authority'].exit_date.max().date()),'sessions':len(books[('candidate',2)]['authority']),'annual_attribution':'trade_date_year'},source_data_as_of=p['as_of'],plan_window_note='start/end inherited from upstream source context; evaluation_first_trade/evaluation_last_trade define evaluation, verified against native account');rec.save_objects(artifact_path='analysis',comparison=table,annual=annual,uncertainty=uncertainty,summary=summary);rec.log_metrics(registered_artifact_count=sum(f.is_file() for f in (Path(summary['recorder_local_dir'])/'artifacts').rglob('*')))
for name,value in [('backtest_benchmark_comparison',table),('annual_metrics',annual),('uncertainty',uncertainty),('training_audit',pd.DataFrame(audits))]:value.to_csv(card/(name+'.csv'),index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':str(provider),'artifact_policy':policy_meta}));binding=json.loads((root/p['hour_source']['card']/'nas_binding.json').read_text());binding.pop('evidence',None);(card/'nas_binding.json').write_text(json.dumps(binding,indent=2));a=Path(summary['recorder_local_dir'])/'artifacts';(card/'verification_manifest.json').write_text(json.dumps({'recorder_id':rec.id,'entries':[{'path':str(f.relative_to(card)),'sha256':hashlib.sha256(f.read_bytes()).hexdigest()} for f in sorted(a.rglob('*')) if f.is_file()]},indent=2));summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));(card/'summary.json').write_text(json.dumps(summary,indent=2));print(pd.DataFrame(audits).to_string(index=False));print(table[['family','slippage_bps','return_pct','monitored_drawdown_pct','recovery_actions']].to_string(index=False));print('warm_difference',warm_difference)


# 补齐最早可用历史后的恢复检验：覆盖改善，策略仍未达标

完成6次拟合、7套新增Qlib原生账户：3个2018配置模型、3个年度恢复模型，1个训练参考账户和0/2/5bp六个配对评价账户。最早范围由冻结价格源自身确定，新增2018-10-15至2018-12-31全部53个可用交易日；没有按收益挑日期或仅取足够达到20条的样本。此前2019及2020–2022原始配置预测全部复用，未重训或重选。

2018配置训练沿用原39输入、20行隔离、300轮和base_seed+年度规则；截止2017-11-30，标签严格成熟于2018-01-01之前，辅助样本为0。2018–2022共1061日训练参考连续平滑、连续持仓和风险状态，2bp费用；局部恢复标签仍为原beta.6/alpha.25动作的扣费共同次日重置优势。

扩展后恢复模型的成熟样本/正类数为2020年28/13、2021年183/106、2022年210/120，三个年度均满足原20条且双类要求。原C=.3、balanced、random_state=42、阈值.825、风险规则均不变，分别10/17/17次迭代收敛。训练参考从更早起点演化，旧日期的状态/费用也可能受影响，不能把结果归因于9条新增动作的独立作用。

评价期仍为2020-01-02起始10万美元现金，至2023-01-03退出，共756区间。新旧评价目标三权重最大差为0；每个成本分别重建同目标、同起点的无恢复对照。原SOXX价格/费用身份一致，复用其已验证账户。所有新账户经原生Qlib现金、股数、费用与净值对账。

| 滑点 | 新恢复候选净收益 | 无恢复对照 | 上轮部分年度模型 | SOXX | 候选监控最大回撤 | 恢复次数 |
|---|---:|---:|---:|---:|---:|---:|
| 0bp | -5.07% | -5.10% | -5.68% | +42.06% | 32.38% | 58 |
| 2bp | -5.17% | -4.97% | -5.54% | +42.04% | 32.50% | 59 |
| 5bp | -7.04% | -6.09% | -7.21% | +42.01% | 32.64% | 54 |

2bp年度候选：2020 -3.95%（52次恢复），2021 +22.12%（0次），2022 -19.16%（7次）；SOXX分别+53.99%、+42.31%、-35.18%。补齐2020模型使动作实际发生，但没有补回上涨参与损失。0bp相对无恢复仅约+0.03个百分点，2/5bp反而退步；不能只比较上轮较弱候选宣布成功。各成本拥有自己的持仓和风险路径，差异不等于单纯扣费。

净SOXX超额、配对全成本增量、30%回撤、年度/滚动稳定及区块下界门均失败。先前2020缺少合格模型的技术缺口已在该固定历史范围内补上，不能继续以该缺口解释当前失败。新增历史仍是已知旧市场资料，模型/规则已经经过历史研究选择，不是新的独立验证。

决策not_promoted_full_history_recovery。此来源最早边界已用完，固定全历史恢复路线关闭；不再截取不同起点、改变20条门槛、C、概率阈值、beta或训练轮数挽救。后续需要新的、时间隔离后仍有净优势的信息，不是继续堆积恢复动作。当前最好历史参考与完整稳定超额目标仍有差距，生产未改，目标未完成。

执行notebook、全部模型、标签、预测、原生账户及来源身份均在Qlib Recorder，NAS冷恢复逐对象哈希通过。训练/账户完成与数据覆盖改善均不冒充收益目标达成。

元数据澄清：原计划start/end为继承的上游来源窗口，实际评价以evaluation_first_trade/evaluation_last_trade为准。summary.json已新增由原生账簿核对的evaluation_window：首交易2020-01-02、末交易2022-12-30、末退出2023-01-03，共756区间；as_of表示可用来源快照日期，不是评价结束日期。原冻结计划、模型、账户、数值结果及NAS归档未改。
